# Try shohon-bbox on one image

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nakamura196/shohon-framing-bbox/blob/main/notebooks/try_in_colab.ipynb)

This notebook draws boxes for the book, colour chart, ruler and imaging slip on one IIIF Canvas from the Kokusho Database (National Institute of Japanese Literature). The example image is licensed CC BY 4.0.

It calls the same command as the paper (`shohon-bbox detect`), so the rules are not repeated here. Library versions on Colab differ from the paper's, so boxes and scores can differ slightly from those in `data/predictions/`.

In [ ]:
!pip install -q "shohon-bbox[detect] @ git+https://github.com/nakamura196/shohon-framing-bbox"

Detect boxes on Canvas 2 of the manifest. The first run downloads the three models (about 1 GB).

In [ ]:
MANIFEST = "https://kokusho.nijl.ac.jp/biblio/300120319/manifest"
CANVAS = "2"
!shohon-bbox detect {MANIFEST} --canvas {CANVAS} > result.json

import json
result = json.load(open("result.json"))
for b in result["boxes"]:
    print(b["class"], b["score"], b["fragment"])

Draw the boxes on the image.

In [ ]:
from PIL import ImageDraw
from shohon_bbox import iiif

im, target = iiif.load(MANIFEST, CANVAS)
sx, sy = im.width / result["width"], im.height / result["height"]
# the colours of the paper's Figure 1
colours = {"book": "#0055ff", "colour_chart": "#00a83f", "ruler": "#e61919", "imaging_slip": "#ea6a00"}
draw = ImageDraw.Draw(im)
for b in result["boxes"]:
    x, y, w, h = b["xywh"]
    draw.rectangle([x * sx, y * sy, (x + w) * sx, (y + h) * sy], outline=colours[b["class"]], width=6)
im.reduce(2)

The same boxes as a IIIF / W3C Web Annotation page, which IIIF viewers can display:

In [ ]:
!shohon-bbox detect {MANIFEST} --canvas {CANVAS} --format annotation > annotations.json
print(open("annotations.json").read()[:800])

To use your own image, replace `MANIFEST` with an image URL or upload a file and give its path (and drop `--canvas`).